# quantum-cortex — the recurrent base hybrid, stage A: references at 3000 steps, then the decisive arms (bare, critical) at 8 pairs (RESUMABLE)

Rev38, staged after the Σ (RESULTS row 27): at 1500 steps a floor is not a verdict, so the arms are compared at 3000 steps against references measured at 3000. Stage A, this session: **the references** (L3 pure on the five seeds the Σ did not run, and the transformer control on the eight, at 8 pairs; the Σ's six units are reused, not recomputed), then **the two decisive arms**, `RB-bare` (the direct measure of pre emption) and `RB-critical` (the founder's arm), eight seeds, 8 pairs. Stage B (a later session): `RB-cls` and `RB-cls-phi`, then 16 pairs. References are matched by unit id (path, seed, tier); only 3000 step directories are passed, never the 1500 step ladder units. A tier subset is a subset aggregate: the paired tests are on seed, at identical steps and tier.

GPU T4, Internet On, about 6 h 30 (13 reference units at about 10 min, 16 arm units at about 16 min); fits one session with `--time-budget-min 700`. With the secrets `KAGGLE_USERNAME` and `KAGGLE_KEY` attached, every finished unit is pushed to a private Kaggle dataset (the relay) and a relaunch in a fresh container pulls them back: nothing is recomputed. Without them, mount this version as an Input and rerun: finished units are reused. **Download the log as a file before reading any number.**

In [ ]:
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("GITHUB_TOKEN")
!rm -rf quantum-cortex
!git clone https://{token}@github.com/vaneeckhoutnicolas/quantum-cortex.git
%cd quantum-cortex
!pip -q install numpy jsonschema


In [ ]:
# the checkpoint relay (optional): with the secrets KAGGLE_USERNAME and KAGGLE_KEY, every finished unit is pushed to a private
# Kaggle dataset and a relaunch in a fresh container pulls them back before computing anything
import os
from kaggle_secrets import UserSecretsClient
try:
    os.environ['KAGGLE_USERNAME'] = UserSecretsClient().get_secret('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = UserSecretsClient().get_secret('KAGGLE_KEY')
    from cortex_data import relay
    print('relay:', 'on' if relay.available() else 'off (kaggle command missing)')
except Exception:
    print('relay: off (secrets KAGGLE_USERNAME / KAGGLE_KEY not attached)')


In [ ]:
# Previous checkpoint dirs mounted as Inputs (resume). The sigma units are in the clone itself.
import glob, os
PREV_REF = sorted(set(glob.glob('/kaggle/input/**/ref3000-ckpt', recursive=True)))
PREV_RB = sorted(set(glob.glob('/kaggle/input/**/rb3000-ckpt', recursive=True)))
print('resume references from:', PREV_REF or '(none — fresh start)')
print('resume arms from:', PREV_RB or '(none — fresh start)')


In [ ]:
# Stage A.1: the references at 3000 steps, 8 pairs: L3 pure on the five other seeds and the control on the eight;
# the sigma's six L3 units (seeds 7, 42, 2024, both tiers) are reused from the clone.
resume = ' '.join(f'--resume-from {p}' for p in ['metrics/mqar/sigma-l3-3000-ckpt'] + PREV_REF)
!python -m cortex_eval.resumable_ladder --paths L3-+local-conv ARCH-none --steps 3000 --seeds 1337 2024 7 42 99 3 11 2026 --tiers 8x128 --ckpt-dir metrics/mqar/ref3000-ckpt --time-budget-min 200 --relay quantum-cortex-relay-ref3000 {resume}


In [ ]:
# Stage A.2: the two decisive arms at 3000 steps, 8 pairs, paired on seed against the references above.
resume = ' '.join(f'--resume-from {p}' for p in PREV_RB)
!python -m cortex_eval.resumable_rb --arms RB-bare RB-critical --steps 3000 --seeds 1337 2024 7 42 99 3 11 2026 --tiers 8x128 --ckpt-dir metrics/mqar/rb3000-ckpt --relay quantum-cortex-relay-rb3000 --reference-from metrics/mqar/ref3000-ckpt metrics/mqar/sigma-l3-3000-ckpt --time-budget-min 480 {resume}


In [ ]:
import glob, json
for p in sorted(glob.glob('metrics/mqar/ref3000-ckpt/*.json')) + sorted(glob.glob('metrics/mqar/rb3000-ckpt/*.json')):
    if 'unit-' in p: continue
    print(p); r = json.load(open(p)); print(json.dumps({k: r[k] for k in r if k in ('stats', 'paired_tests', 'gate_readout', 'references', 'complete', 'units_done', 'units_total')}, indent=1)[:4000])
print('Commit metrics/mqar/ref3000-ckpt/ and metrics/mqar/rb3000-ckpt/ — the units are the record.')
